<a href="https://colab.research.google.com/github/davide-creator/project1/blob/main/X.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import re
import sys
import zipfile
import pandas as pd
import win32com.client
import PyPDF2
from openpyxl import load_workbook

# -- CONFIGURAZIONE / LETTURA FILE EXCEL -------------------------------------

# Imposta qui il percorso del file Excel con le variabili di configurazione
path_excel = r"C:\Percorso\AlTuoFile\Project Xhina & Priyanka\variables.xlsx"
# Imposta qui il percorso del file di testo che conterrà gli ID delle mail già processate
processed_emails_file = r"C:\Percorso\AlTuoFile\Project Xhina & Priyanka\processed_emails.txt"

# Legge il file di configurazione Excel (colonna 0 = chiave, colonna 1 = valore)
df = pd.read_excel(path_excel, header=None, index_col=0)
config = {str(k).strip(): str(v).strip() for k, v in df[1].items()}

# -- FUNZIONI PER RIMOZIONE PASSWORD / ESTRAZIONE ----------------------------

def remove_pdf_password(input_path, output_path, pdf_password):
    """
    Apre un PDF potenzialmente protetto da password e lo riscrive
    senza password nel file `output_path`.
    """
    try:
        with open(input_path, 'rb') as in_file:
            reader = PyPDF2.PdfReader(in_file)
            # Se il PDF è crittografato, prova a decriptarlo
            if reader.is_encrypted:
                reader.decrypt(pdf_password)
            writer = PyPDF2.PdfWriter()

            # Copia tutte le pagine nel nuovo PDF non protetto
            for page_num in range(len(reader.pages)):
                writer.add_page(reader.pages[page_num])

            with open(output_path, 'wb') as out_file:
                writer.write(out_file)

        print(f"[OK] PDF decrittato: {input_path} -> {output_path}")
    except Exception as e:
        print(f"[ERRORE] Rimozione password PDF fallita per '{input_path}': {e}")

def remove_excel_password(input_path, output_path):
    """
    Stub di esempio. La rimozione password dagli Excel .xlsx può variare
    a seconda della protezione usata (strutturale, protezione foglio, ecc.).
    Qui è lasciato come esempio da personalizzare se necessario.
    """
    try:
        # Esempio semplicissimo: copiare il file come "non protetto".
        # In realtà, a seconda del tipo di protezione, potrebbe servire
        # un approccio diverso o una libreria dedicata.
        with open(input_path, 'rb') as f_in, open(output_path, 'wb') as f_out:
            f_out.write(f_in.read())
        print(f"[OK] Excel copiato (password removal da implementare): {input_path} -> {output_path}")
    except Exception as e:
        print(f"[ERRORE] Rimozione password Excel fallita per '{input_path}': {e}")


def process_zip_file(zip_file_path, extract_parent_dir):
    """
    Estrae il contenuto di uno ZIP in una cartella con lo stesso nome del file ZIP,
    senza utilizzare alcuna password. Poi elabora eventuali PDF/XLSX/ZIP presenti
    nella cartella estratta.
    """
    base_name = os.path.splitext(os.path.basename(zip_file_path))[0]
    target_folder = os.path.join(extract_parent_dir, base_name)
    os.makedirs(target_folder, exist_ok=True)

    try:
        # Estrazione standard (senza password)
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(target_folder)

        print(f"[OK] ZIP estratto: '{zip_file_path}' -> Cartella: {target_folder}")
    except Exception as e:
        print(f"[ERRORE] Durante l'estrazione di '{zip_file_path}': {e}")
        return

    # Ora scansioniamo i file estratti e applichiamo eventuale logica (PDF, ZIP annidati, ecc.)
    for root, dirs, files in os.walk(target_folder):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            # Se è un PDF, rimuovi password
            if file_name.lower().endswith('.pdf'):
                pdf_no_pass_path = file_path.replace('.pdf', '_nopass.pdf')
                remove_pdf_password(file_path, pdf_no_pass_path, config.get('pdf_password', ''))
            # Se è un .zip annidato, processalo ricorsivamente
            elif file_name.lower().endswith('.zip'):
                process_zip_file(file_path, root)
            # Se è un Excel, rimuovi password (se implementato)
            elif file_name.lower().endswith('.xlsx'):
                excel_no_pass_path = file_path.replace('.xlsx', '_nopass.xlsx')
                remove_excel_password(file_path, excel_no_pass_path)


# -- FUNZIONI PER GESTIONE EMAIL ---------------------------------------------

def load_processed_emails(file_path):
    """Carica la lista di email già elaborate da un file di testo."""
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            return [line.strip() for line in f if line.strip()]
    else:
        return []

def save_processed_email(email_id, file_path):
    """Salva l'ID di un'email elaborata in un file di testo, per non riprocessarla."""
    with open(file_path, 'a', encoding='utf-8') as f:
        f.write(email_id + '\n')

def process_emails():
    """Funzione principale che si collega a Outlook, individua gli allegati e li elabora."""
    pdf_password = config.get('pdf_password', '')
    save_path = config.get('save_path', r"C:\Temp")
    outlook_folder_name = config.get('outlook_folder', 'Inbox')

    # Avvia l'applicazione Outlook
    outlook = win32com.client.Dispatch('Outlook.Application')
    session = outlook.GetNameSpace('MAPI')

    # Trova la cartella desiderata (tra i vari account di Outlook)
    inbox = None
    for store in session.Folders:
        try:
            candidate = store.Folders[outlook_folder_name]
            if candidate:
                inbox = candidate
                break
        except:
            pass

    if not inbox:
        print(f"[ERRORE] La cartella '{outlook_folder_name}' non è stata trovata in Outlook.")
        return

    # Carica la lista di email già processate
    processed_emails = load_processed_emails(processed_emails_file)

    # Scansione di tutti i messaggi
    for message in inbox.Items:
        # Verifica se l'ID è già in elenco
        if message.EntryID in processed_emails:
            continue  # Salta email già processata

        # Gestione allegati
        if message.Attachments.Count > 0:
            for attachment in message.Attachments:
                attachment_name = attachment.FileName
                attachment_path = os.path.join(save_path, attachment_name)

                # Salva fisicamente l'allegato
                try:
                    attachment.SaveASFile(attachment_path)
                except Exception as e:
                    print(f"[ERRORE] Non riesco a salvare l'allegato '{attachment_name}': {e}")
                    continue

                # In base all'estensione, decidi cosa fare
                if attachment_name.lower().endswith('.pdf'):
                    # Rimuove password dal PDF (se c'è)
                    out_pdf = attachment_path.replace('.pdf', '_nopassword.pdf')
                    remove_pdf_password(attachment_path, out_pdf, pdf_password)

                elif attachment_name.lower().endswith('.xlsx'):
                    # Rimuove password dal file Excel (se implementato)
                    out_xlsx = attachment_path.replace('.xlsx', '_nopassword.xlsx')
                    remove_excel_password(attachment_path, out_xlsx)

                elif attachment_name.lower().endswith('.zip'):
                    # Estrae il file zip in una cartella omonima
                    process_zip_file(attachment_path, save_path)

        # Segna l'email come processata
        save_processed_email(message.EntryID, processed_emails_file)

# -- MAIN --------------------------------------------------------------------

if __name__ == "__main__":
    process_emails()